In [0]:
from pyspark.sql.functions import col

listings_silver = spark.table("airbnb_barcelona_project.silver.listings")
fact_listings = spark.table("airbnb_barcelona_project.gold.fact_listings")
dim_host = spark.table("airbnb_barcelona_project.gold.dim_host")
dim_neighbourhood = spark.table("airbnb_barcelona_project.gold.dim_neighbourhood")

print("=" * 60)
print("1. ROW COUNT RECONCILIATION")
print("=" * 60)
bronze_count = spark.table("airbnb_barcelona_project.bronze.listings").count()
silver_count = listings_silver.count()
gold_count = fact_listings.count()
print(f"Bronze: {bronze_count}, Silver: {silver_count}, Gold fact: {gold_count}")

1. ROW COUNT RECONCILIATION
Bronze: 15293, Silver: 15293, Gold fact: 15293


In [0]:
print("=" * 60)
print("2. DUPLICATE KEY CHECK")
print("=" * 60)
total = fact_listings.count()
distinct = fact_listings.select("id").distinct().count()
print(f"fact_listings: {total} rows, {distinct} distinct id ({'OK' if total == distinct else 'DUPLICATES FOUND'})")

2. DUPLICATE KEY CHECK
fact_listings: 15293 rows, 15293 distinct id (OK)


In [0]:
print("=" * 60)
print("3. REFERENTIAL INTEGRITY — fact vs dimensions")
print("=" * 60)
orphans_neighbourhood = fact_listings.join(dim_neighbourhood, "neighbourhood_cleansed", "left_anti")
print(f"Listings with no matching neighbourhood: {orphans_neighbourhood.count()} (should be 0)")

orphans_host = fact_listings.join(dim_host, "host_id", "left_anti")
print(f"Listings with no matching host: {orphans_host.count()} (should be 0)")

3. REFERENTIAL INTEGRITY — fact vs dimensions
Listings with no matching neighbourhood: 0 (should be 0)
Listings with no matching host: 0 (should be 0)


In [0]:
print("=" * 60)
print("4. NULL CHECKS ON KEY COLUMNS")
print("=" * 60)
for c in ["id", "host_id", "neighbourhood_cleansed", "room_type"]:
    nulls = fact_listings.filter(col(c).isNull()).count()
    print(f"  {c}: {nulls} nulls")

4. NULL CHECKS ON KEY COLUMNS
  id: 0 nulls
  host_id: 0 nulls
  neighbourhood_cleansed: 0 nulls
  room_type: 0 nulls


In [0]:
print("=" * 60)
print("5. PRICE SANITY CHECK")
print("=" * 60)
negative_or_zero = fact_listings.filter((col("price") <= 0)).count()
print(f"Listings with price <= 0 (excluding nulls): {negative_or_zero} (should be 0)")
fact_listings.select("price").summary("count", "mean", "min", "max").show()

5. PRICE SANITY CHECK
Listings with price <= 0 (excluding nulls): 0 (should be 0)
+-------+----------------+
|summary|           price|
+-------+----------------+
|  count|           13355|
|   mean|240.616470235866|
|    min|             3.5|
|    max|        10542.22|
+-------+----------------+



In [0]:
c# Look at the cheapest listings
display(fact_listings.filter(col("price") < 15).select("id", "neighbourhood_cleansed", "room_type", "price", "accommodates").orderBy("price").limit(15))

# Look at the most expensive listings
display(fact_listings.filter(col("price") > 2000).select("id", "neighbourhood_cleansed", "room_type", "price", "accommodates").orderBy(col("price").desc()).limit(15))

# How many listings fall in these extreme ranges?
print(f"Listings under €15/night: {fact_listings.filter(col('price') < 15).count()}")
print(f"Listings over €2000/night: {fact_listings.filter(col('price') > 2000).count()}")

id,neighbourhood_cleansed,room_type,price,accommodates
38039288,el Poble Sec,Private room,3.5,1
906440016225374191,la Barceloneta,Entire home/apt,6.88,2
31249193,el Camp de l'Arpa del Clot,Entire home/apt,9.07,3
913102185439858883,el Camp de l'Arpa del Clot,Private room,9.18,1
675175,la Prosperitat,Private room,9.62,1
51455847,la Barceloneta,Entire home/apt,9.85,2
46052188,la Dreta de l'Eixample,Private room,10.95,2
1207731279848423979,el Guinardó,Entire home/apt,11.06,2
31203362,el Coll,Private room,11.85,1
1070996015283010717,el Camp de l'Arpa del Clot,Private room,11.96,1


id,neighbourhood_cleansed,room_type,price,accommodates
867256504774715226,el Barri Gòtic,Entire home/apt,10542.22,2
1426637760045243370,la Dreta de l'Eixample,Entire home/apt,7532.0,16
1405062163700981309,la Dreta de l'Eixample,Entire home/apt,7293.0,16
1406454143054865483,la Dreta de l'Eixample,Entire home/apt,6151.0,16
1400057633003120901,la Dreta de l'Eixample,Entire home/apt,6106.0,16
1157205495486273795,la Dreta de l'Eixample,Entire home/apt,5900.0,16
1309611159285218484,la Dreta de l'Eixample,Entire home/apt,4885.0,16
909087702399642381,la Nova Esquerra de l'Eixample,Entire home/apt,4463.0,4
1308738273720498469,la Nova Esquerra de l'Eixample,Entire home/apt,4394.0,16
1473123174977324233,"Sant Pere, Santa Caterina i la Ribera",Entire home/apt,4080.0,16


Listings under €15/night: 26
Listings over €2000/night: 98


In [0]:
# Is this really 98 distinct properties, or repeats/variations of a few?
display(fact_listings.filter((col("price") > 2000) & (col("accommodates") == 16))
        .select("neighbourhood_cleansed", "price").orderBy("neighbourhood_cleansed"))

print(f"Distinct host_ids among >€2000 listings: {fact_listings.filter(col('price') > 2000).select('host_id').distinct().count()}")

neighbourhood_cleansed,price
Hostafrancs,3504.0
Hostafrancs,2443.0
Sant Antoni,2557.0
Sant Antoni,2774.0
Sant Antoni,3698.0
Sant Antoni,2317.0
Sant Antoni,2602.0
Sant Antoni,3356.0
Sant Antoni,3196.0
Sant Antoni,2557.0


Distinct host_ids among >€2000 listings: 10


In [0]:
from pyspark.sql.functions import col

# Flag extreme outliers rather than deleting — same philosophy as Olist/NYC311 quarantine pattern
fact_listings_flagged = fact_listings.withColumn(
    "price_outlier",
    (col("price") < 15) | (col("price") > 2000)
)

fact_listings_flagged.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("airbnb_barcelona_project.gold.fact_listings")

outlier_count = fact_listings_flagged.filter(col("price_outlier") == True).count()
print(f"Flagged as price outliers: {outlier_count} of {fact_listings_flagged.count()} ({round(outlier_count/fact_listings_flagged.count()*100,2)}%)")


Flagged as price outliers: 124 of 15293 (0.81%)


In [0]:
dim_host = spark.table("airbnb_barcelona_project.gold.dim_host")
print(f"Rows: {dim_host.count()}, Distinct host_id: {dim_host.select('host_id').distinct().count()}")

Rows: 4595, Distinct host_id: 4595
